### py4cytoscape

https://py4cytoscape.readthedocs.io/en/latest/

In [ ]:
from pathlib import Path
from rdflib import Graph, RDF, Namespace
import networkx as nx
import py4cytoscape as p4c
import inspect

In [ ]:
print(inspect.signature(p4c.set_node_shape_mapping))
print(inspect.signature(p4c.set_node_color_mapping))

### Cytoscape 

/home/flavio/Cytoscape_v3.10.2

cd /media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/cytoscape
wget https://github.com/cytoscape/cytoscape/releases/download/3.10.2/Cytoscape_3_10_2_unix.sh

#### make executable
chmod +x Cytoscape_3_10_2_unix.sh

#### java 
sudo apt update
sudo apt install openjdk-17-jdk
java -version
whereis java
readlink -f /usr/bin/java
java -version

####  openjdk 17.0.18 2026-01-20
#### OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
#### OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)

#### install
./Cytoscape_3_10_2_unix.sh

#### run
> export INSTALL4J_JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64  
> ~/Cytoscape_v3.10.2/Cytoscape &


In [ ]:
root_owl = Path('../owl/')
BP = Namespace("http://www.biopax.org/release/biopax-level3.owl#")

G = nx.DiGraph()

classes = [
    BP.Pathway,
    BP.BiochemicalReaction,
    BP.Catalysis,
    BP.Control,
    BP.Protein,
    BP.Complex,
    BP.SmallMolecule,
    BP.PhysicalEntity,
]

relations = [
    BP.pathwayComponent,
    BP.left,
    BP.right,
    BP.controller,
    BP.controlled,
    BP.participant,
    BP.component,
]


def read_owl(owl_file):
    fnameowl = root_owl / owl_file

    rdf = Graph()
    rdf.parse(fnameowl, format="xml")

    return rdf

def short(x):
    return str(x).split("#")[-1].split("/")[-1]

def get_name(x, rdf):
    for prop in [BP.displayName, BP.standardName, BP.name]:
        value = next(rdf.objects(x, prop), None)
        if value:
            return str(value)
    return short(x)

In [ ]:
owl_file = "R-HSA-165159_level3.owl"
owl_file = "R-HSA-114608_level3.owl"

rdf = read_owl(owl_file)

In [ ]:
for cls in classes:
    for node in rdf.subjects(RDF.type, cls):
        node_id = short(node)
        G.add_node(node_id, label=get_name(node, rdf), biopax_type=short(cls))

In [ ]:
for rel in relations:
    for s, o in rdf.subject_objects(rel):
        s_id = short(s)
        o_id = short(o)

        if s_id in G.nodes and o_id in G.nodes:
            G.add_edge(s_id, o_id, interaction=short(rel))

In [ ]:
p4c.cytoscape_ping()

p4c.create_network_from_networkx(G, title=f"Reactome {owl_file}", collection="Reactome OWL")

In [ ]:
style_name = "Reactome_BioPAX_Style"

p4c.create_visual_style(style_name)

p4c.set_node_label_mapping("label", style_name=style_name)

p4c.set_node_color_mapping(
    "biopax_type",
    ["Protein", "Complex", "SmallMolecule", "BiochemicalReaction", "Pathway"],
    ["#8ecae6", "#ffb703", "#90be6d", "#f94144", "#cdb4db"],
    mapping_type="d",
    default_color="#cccccc",
    style_name=style_name
)

p4c.set_node_shape_mapping(
    "biopax_type",
    ["Protein", "Complex", "SmallMolecule", "BiochemicalReaction", "Pathway"],
    ["ELLIPSE", "ROUND_RECTANGLE", "DIAMOND", "HEXAGON", "RECTANGLE"],
    default_shape="ELLIPSE",
    style_name=style_name
)

p4c.set_visual_style(style_name)
p4c.layout_network("force-directed")

```Python
G.nodes[gene_id]["log2FC"] = log2fc
G.nodes[gene_id]["FDR"] = fdr
G.nodes[gene_id]["mutated"] = True
```

### OWL to Graph

In [ ]:
def nx_to_cytoscape_elements(G, saved_positions=None):
    elements = []

    saved_positions = saved_positions or {}

    for node_id, data in G.nodes(data=True):
        elem = {
            "data": {
                "id": node_id,
                "label": data.get("label", node_id),
                "biopax_type": data.get("biopax_type", "Unknown"),
                "log2FC": data.get("log2FC", None),
                "FDR": data.get("FDR", None),
            }
        }

        if node_id in saved_positions:
            elem["position"] = saved_positions[node_id]

        elements.append(elem)

    for source, target, data in G.edges(data=True):
        elements.append({
            "data": {
                "source": source,
                "target": target,
                "interaction": data.get("interaction", "")
            }
        })

    return elements

In [ ]:
elements = nx_to_cytoscape_elements(G)

### Dash

In [ ]:
import dash
from dash import html, Input, Output, State, dcc
import dash_cytoscape as cyto
import json

In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div([
    cyto.Cytoscape(
        id="reactome-network",
        elements=elements,
        layout={"name": "cose"},
        style={"width": "100%", "height": "800px"},
        stylesheet=[
            {
                "selector": "node",
                "style": {
                    "label": "data(label)",
                    "font-size": "10px",
                    "background-color": "#8ecae6",
                    "width": 30,
                    "height": 30,
                },
            },
            {
                "selector": "edge",
                "style": {
                    "curve-style": "bezier",
                    "target-arrow-shape": "triangle",
                    "label": "data(interaction)",
                    "font-size": "7px",
                },
            },
            {
                "selector": '[biopax_type = "BiochemicalReaction"]',
                "style": {"background-color": "#f94144", "shape": "hexagon"},
            },
            {
                "selector": '[biopax_type = "Protein"]',
                "style": {"background-color": "#8ecae6", "shape": "ellipse"},
            },
            {
                "selector": '[biopax_type = "Complex"]',
                "style": {"background-color": "#ffb703", "shape": "round-rectangle"},
            },
            {
                "selector": '[biopax_type = "SmallMolecule"]',
                "style": {"background-color": "#90be6d", "shape": "diamond"},
            },
            {
                "selector": '[biopax_type = "Pathway"]',
                "style": {"background-color": "#cdb4db", "shape": "rectangle"},
            },
        ],
    ),

    html.Button("Save node positions", id="save-button"),
    html.Pre(id="saved-output"),
])

In [ ]:
@app.callback(
    Output("saved-output", "children"),
    Input("save-button", "n_clicks"),
    State("reactome-network", "elements"),
    prevent_initial_call=True,
)
def save_positions(n_clicks, elements):
    positions = {}

    for elem in elements:
        data = elem.get("data", {})
        pos = elem.get("position")

        if "id" in data and pos is not None:
            positions[data["id"]] = pos

    with open("reactome_saved_positions.json", "w") as f:
        json.dump(positions, f, indent=2)

    return json.dumps(positions, indent=2)

In [ ]:
app.run_server(debug=True, port=8050)